# 🧪 معمل التجارب: بناء وفهرسة المتجهات بمكتبة Meta FAISS (Lecture 32)
أهلاً بك في معملك التطبيقي! 🚀

**فكرة المعمل ببساطة:**
مكتبة **FAISS (Facebook AI Similarity Search)** التي طورتها شركة Meta هي المعيار العالمي الأكثر كفاءة للبحث الدلالي فائق السرعة في الذاكرة.

في هذا الجزء الأول من المحاضرة 32 ستتعلم عملياً:
1. **مقارنة التضمينات:** حساب تشابه جيب التمام (Cosine Similarity) رياضياً بين المفاهيم.
2. **بناء فهرس FAISS في الذاكرة:** تحويل المستندات والـ Metadata إلى فهرس حي سريع.
3. **الحفظ والاسترجاع المحلي:** حفظ الفهرس في ملفي (`index.faiss` و `index.pkl`) وإعادة تحميله بأمان.
4. **أنواع البحث المختلفة:** البحث العادي، والبحث مع درجات مسافة إقليدس (L2 Distance Scores)، والتصفية بالبيانات الوصفية.

> 💡 **طريقة التدريب في المعمل:**
> اقرأ الشرح المبسط ⬅️ شوف الكود المرجعي الجاهز ⬅️ واكتب الكود وجربه بإيدك في الخلية الفارغة المخصصة لك تحت كل خطوة!

### 1. استيراد المكتبات وتحديد مسار قاعدة البيانات المركزي
**الفكرة ببساطة:**
بنستورد مكتبة `FAISS` من لانج تشين ونموذج التضمين ونحدد مسار الحفظ المركزي `data/databases/faiss_index`.

In [ ]:
# 1. import standard libraries and load environment variables
import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# resolve centralized database directory
data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
faiss_db_dir = str(data_dir / "databases" / "faiss_index")

# import langchain components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# initialize local huggingface embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("✅ Libraries imported successfully!")
print(f"📁 Target FAISS Directory: {faiss_db_dir}")

In [1]:
# ✍️ مساحتك للتطبيق العملي (جرب استيراد المكتبات وتجهيز التضمين والمسار بنفسك هنا):
from langchain.chat_models import init_chat_module

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = init_chat_module(
    model_name="groq/groq-llm-13b")

llm


ImportError: cannot import name 'init_chat_module' from 'langchain.chat_models' (d:\Code\AI\rag-mastery\.venv\Lib\site-packages\langchain\chat_models\__init__.py)

### 2. تجهيز المستندات الأولية وإرفاق البيانات الوصفية (Metadata)
**الفكرة ببساطة:**
بننشئ 4 مستندات تمثل مجالات الذكاء الاصطناعي (AI, ML, DL, NLP) ونعطي كل مستند Metadata بتحدد مجاله ومصدره لتمكين التصفية الدقيقة لاحقاً.

In [ ]:
# create raw sample documents with metadata
sample_documents = [
    Document(
        page_content=(
            "Artificial Intelligence (AI) is the simulation of human intelligence by machines. "
            "It encompasses machine learning, deep learning, natural language processing, "
            "and computer vision to build systems capable of rational reasoning, problem solving, "
            "and autonomous decision making."
        ),
        metadata={"topic": "ai", "source": "intro_ai.txt"}
    ),
    Document(
        page_content=(
            "Machine Learning (ML) is a subset of AI where systems learn from historical data "
            "rather than relying on hardcoded procedural rules. Key paradigms include supervised learning "
            "(regression, classification), unsupervised learning (clustering), and reinforcement learning."
        ),
        metadata={"topic": "ml", "source": "ml_basics.txt"}
    ),
    Document(
        page_content=(
            "Deep Learning (DL) is a specialized branch of machine learning based on multi-layered artificial "
            "neural networks. Deep architectures such as Convolutional Neural Networks (CNNs) and Transformers "
            "excel at extracting hierarchical representations from complex visual and textual data."
        ),
        metadata={"topic": "dl", "source": "deep_learning.txt"}
    ),
    Document(
        page_content=(
            "Natural Language Processing (NLP) enables machines to understand, interpret, and generate human language. "
            "Core tasks include syntactic parsing, semantic analysis, sentiment detection, translation, and large language modeling."
        ),
        metadata={"topic": "nlp", "source": "nlp_overview.txt"}
    )
]

print(f"✅ Created {len(sample_documents)} sample documents with metadata.")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب إنشاء المستندات مع الـ metadata بنفسك هنا):



### 3. تقطيع المستندات إلى Chunks متجانسة
**الفكرة ببساطة:**
بنقطع المستندات لقطع بحجم 500 حرف وتداخل 50 حرف، والـ Metadata بتنتقل تلقائياً لكل قطعة ناتجة.

In [ ]:
# configure text splitter
chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len
)

chunked_documents = text_splitter.split_documents(sample_documents)
print(f"✅ Split documents into {len(chunked_documents)} chunks.\n")

for idx, doc in enumerate(chunked_documents, start=1):
    print(f"  [{idx}] (Topic: {doc.metadata['topic']}): {doc.page_content[:70]}...")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تقطيع المستندات إلى chunks بنفسك هنا):



### 4. فحص التضمينات وتطبيق دالة تشابه جيب التمام (Cosine Similarity)
**الفكرة ببساطة:**
بنكتب دالة رياضية بسيطة بتحسب الزاوية بين متجهين:
الكلمات اللي شبه بعض بتدي رقم قريب من 1 (تشابه عالي)، والكلمات اللي ملهاش علاقة ببعض بتدي رقم قريب من الصفر.

In [ ]:
# helper function to compute cosine similarity between two texts
def compare_embeddings(text1: str, text2: str) -> float:
    v1 = np.array(embeddings.embed_query(text1))
    v2 = np.array(embeddings.embed_query(text2))
    cosine_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return float(cosine_sim)

# test semantic similarity comparisons
sim_synonym = compare_embeddings("AI", "Artificial Intelligence")
sim_acronym = compare_embeddings("Machine Learning", "ML")
sim_unrelated = compare_embeddings("AI", "Italian Pizza")

print(f"🔍 Similarity ('AI' vs 'Artificial Intelligence'): {sim_synonym:.4f} (High similarity)")
print(f"🔍 Similarity ('Machine Learning' vs 'ML'):       {sim_acronym:.4f} (High similarity)")
print(f"🔍 Similarity ('AI' vs 'Italian Pizza'):           {sim_unrelated:.4f} (Low similarity)")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب كتابة دالة حساب التشابه ومقارنة كلمات مختلفة بنفسك هنا):



### 5. إنشاء فهرس المتجهات بمكتبة FAISS في الذاكرة (`FAISS.from_documents`)
**الفكرة ببساطة:**
بنحول القطع لمتجهات ونبني الفهرس في الذاكرة بـ `FAISS.from_documents` ونتأكد من عدد المتجهات بـ `index.ntotal`.

In [ ]:
# build FAISS vector store from document chunks
vector_store = FAISS.from_documents(chunked_documents, embeddings)

print("✅ FAISS vector store created successfully in memory!")
print(f"📊 Total indexed vectors (ntotal): {vector_store.index.ntotal}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب بناء فهرس FAISS وفحص ntotal بنفسك هنا):



### 6. حفظ الفهرس على القرص وإعادة تحميله محلياً (`save_local` & `load_local`)
**الفكرة ببساطة:**
بنحفظ الفهرس في مجلد `data/databases/faiss_index`، فبينتج ملفين: `index.faiss` للمتجهات الرقمية، و `index.pkl` للنصوص والبيانات الوصفية.
ولما بنحمله بـ `load_local` بنمرر `allow_dangerous_deserialization=True` عشان يقرا ملف الـ Pickle بأمان.

In [ ]:
# 1. save FAISS index locally to disk
vector_store.save_local(faiss_db_dir)
saved_files = [f.name for f in Path(faiss_db_dir).iterdir()]
print(f"💾 FAISS index saved to: {faiss_db_dir}")
print(f"📄 Saved files: {saved_files}\n")

# 2. reload the index from disk
loaded_vector_store = FAISS.load_local(
    faiss_db_dir,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"✅ Successfully reloaded FAISS index! Total vectors: {loaded_vector_store.index.ntotal}")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب حفظ الفهرس وإعادة تحميله من القرص بنفسك هنا):



### 7. البحث الدلالي العادي (Similarity Search)
**الفكرة ببساطة:**
بنسأل سؤال دلالي بـ `.similarity_search(query, k=3)` و FAISS بتجيب أقرب 3 قطع لمعنى السؤال.

In [ ]:
# perform standard similarity search
query = "What is deep learning and how does it relate to neural networks?"
results = loaded_vector_store.similarity_search(query, k=3)

print(f"❓ Query: '{query}'\n")
print(f"🔍 Top {len(results)} Similar Chunks:\n")
for idx, doc in enumerate(results, start=1):
    print(f"  [{idx}] (Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}):")
    print(f"      {doc.page_content[:95]}...\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب تشغيل البحث الدلالي العادي بنفسك هنا):



### 8. البحث مع استخراج درجات المسافة الإقليدية (L2 Distance Scores)
**الفكرة ببساطة:**
مقياس FAISS الافتراضي هو مسافة L2 الإقليدية:
**الرقم الأصغر يعني قرب وتطابق أعلى** (المسافة صفر = تطابق كامل 100%).

In [ ]:
# similarity search returning (document, L2_distance) pairs
results_with_scores = loaded_vector_store.similarity_search_with_score(query, k=3)

print(f"❓ Query: '{query}'\n")
print("📊 Results with L2 Distance Scores (LOWER score = MORE similar):\n")
for idx, (doc, score) in enumerate(results_with_scores, start=1):
    print(f"  Rank [{idx}] -> L2 Distance: {score:.4f}")
    print(f"    Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"    Content: {doc.page_content[:90]}...\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب استخراج درجات المسافة L2 وفحص النتيجة بنفسك هنا):



### 9. البحث مع التصفية بالبيانات الوصفية (Metadata Filtering)
**الفكرة ببساطة:**
بنحدد شرط تصفية `filter={"topic": "ml"}`، فتقوم FAISS بحصر البحث في مستندات الـ Machine Learning فقط واستبعاد أي مجال آخر.

In [ ]:
# similarity search with metadata filter
filter_dict = {"topic": "ml"}
filtered_results = loaded_vector_store.similarity_search(
    query="Explain key foundational concepts",
    k=3,
    filter=filter_dict
)

print(f"🔍 Search with Filter: {filter_dict}")
print(f"✅ Returned {len(filtered_results)} matching chunks:\n")
for idx, doc in enumerate(filtered_results, start=1):
    print(f"  [{idx}] Topic: {doc.metadata['topic']} | Source: {doc.metadata['source']}")
    print(f"      Content: {doc.page_content[:95]}...\n")

In [ ]:
# ✍️ مساحتك للتطبيق العملي (جرب البحث مع فلترة metadata بنفسك هنا):



## 🎯 ملخص المعمل السريع (Lecture 32):
1. **Meta FAISS:** أقوى مكتبة مفتوحة المصدر للبحث عالي السرعة في المتجهات بالذاكرة.
2. **ملفا الحفظ:** `save_local` بتنشئ `index.faiss` للمتجهات الثنائية و `index.pkl` للنصوص والـ Metadata.
3. **مسافة L2:** المسافة الأقل تعني تشابهاً وتطابقاً أعلى.
4. **المحطة القادمة (Lecture 33):** تحويل فهرس FAISS لشات بوت RAG متكامل بـ LCEL والبث اللحظي والذاكرة!